# KMeans Clustering: Synthetic Blobs and Wine Dataset

This notebook demonstrates KMeans clustering with:
1. Synthetic data (make_blobs) to understand the algorithm
2. Real-world Wine dataset clustering
3. Elbow method and silhouette analysis for choosing k
4. PCA visualization of clusters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, load_wine
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

sns.set_style('whitegrid')
%matplotlib inline

## 1. KMeans on Synthetic Data

We start with `make_blobs` to create clearly separated clusters, verifying KMeans can recover the true structure.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=500, centers=4, cluster_std=0.8, random_state=42)

kmeans_blobs = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_blobs = kmeans_blobs.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='Set2', s=20, alpha=0.7)
axes[0].set_title('True Labels')

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_blobs, cmap='Set2', s=20, alpha=0.7)
axes[1].scatter(kmeans_blobs.cluster_centers_[:, 0], kmeans_blobs.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[1].set_title('KMeans Predicted Labels')

plt.tight_layout()
plt.show()
print(f"Silhouette Score: {silhouette_score(X_blobs, labels_blobs):.4f}")

## 2. Wine Dataset: Finding Natural Clusters

The Wine dataset contains chemical analysis results of wines from three cultivars. Let's see if KMeans can discover these groups.

In [ ]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
print(f"Shape: {df.shape}")
print(f"True classes: {wine.target_names}")
df.describe().round(2)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(wine.data)

## 3. Elbow Method and Silhouette Analysis

The **elbow method** plots inertia (within-cluster sum of squares) vs. k. The **silhouette score** measures how well each point fits its cluster vs. the nearest neighboring cluster.

In [ ]:
K_range = range(2, 11)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(K_range, sil_scores, 'rs-', linewidth=2)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')

plt.tight_layout()
plt.show()
print(f"Best k by silhouette: {K_range[np.argmax(sil_scores)]} (score: {max(sil_scores):.4f})")

## 4. PCA Visualization of Clusters

We project the scaled data to 2D using PCA and compare KMeans clusters (k=3) against true wine cultivar labels.

In [ ]:
kmeans_wine = KMeans(n_clusters=3, random_state=42, n_init=10)
wine_labels = kmeans_wine.fit_predict(X_scaled)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
centers_pca = pca.transform(kmeans_wine.cluster_centers_)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=wine.target, cmap='Set1', s=30, alpha=0.7)
axes[0].set_title('True Wine Cultivar Labels')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=wine_labels, cmap='Set1', s=30, alpha=0.7)
axes[1].scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', marker='X', s=200, edgecolors='white', linewidths=2)
axes[1].set_title('KMeans Clusters (k=3)')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')

plt.tight_layout()
plt.show()
print(f"Silhouette Score (k=3): {silhouette_score(X_scaled, wine_labels):.4f}")

In [ ]:
# Compare different k values side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, k in zip(axes, [2, 3, 5]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='Set2', s=30, alpha=0.7)
    ax.set_title(f'k={k} (silhouette={sil:.3f})')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.tight_layout()
plt.show()